In [ ]:
import legion_goes
print(legion_goes.__version__)  # 0.1.9

from legion_goes.sot.goes_hardcoded.core.goes_info_sat import AVAILABLE_GOES_ID
print(AVAILABLE_GOES_ID)

In [ ]:
import os

# Obtener el path actual
cwd = os.getcwd()

print(f"📍 Directorio de trabajo actual: {cwd}")

In [ ]:
# =============================================================================
# FILE PATH: legion_goes/tasks/task03_processing/subtask01_proc_single/task03_processing_subtask01_proc_single.py
# Version: 1.0.0 (The Trinity: Generate -> Audit -> Run)
# =============================================================================

import time

# --- IMPORTACIÓN DE ACCIONES ---
from legion_goes.tasks.task03_processing.subtask01_proc_single.actions.action01_generate_plan_proc_single import run_action as action_01
from legion_goes.tasks.task03_processing.subtask01_proc_single.actions.action02_update_plan_audit_files import run_action as action_02
from legion_goes.tasks.task03_processing.subtask01_proc_single.actions.action03_run_executor_proc_single import run_action as action_03

def run_subtask_01(sat_id: str, product_id: str, year: str, day: str, fnp_tag: str = "fnp01") -> bool:
    """
    Orquestador Maestro: Ejecuta el pipeline completo de planificación y procesamiento.
    """
    ctx = "[SUBTASK 01 - MASTER]"
    start_all = time.time()
    
    print("\n" + f" INICIANDO PROCESAMIENTO COMPLETO: {product_id} G{sat_id} ".center(80, "="))
    print(f"📅 Fecha: Year {year}, Day {day} | Tag: {fnp_tag}")

    try:
        # --- PASO 1: GENERAR PLAN (THEORETICAL) ---
        print(f"\n🔹 [PASO 1/3] Generando/Verificando Plan JSON...")
        if not action_01(sat_id, product_id, year, day, fnp_tag):
            print(f"❌ {ctx} Falló la creación del plan. Abortando.")
            return False
        
        # --- PASO 2: AUDITAR DISCO (REALITY CHECK) ---
        print(f"\n🔹 [PASO 2/3] Auditando archivos en disco...")
        if not action_02(sat_id, product_id, year, day, fnp_tag):
            print(f"❌ {ctx} Falló la auditoría de archivos. Abortando.")
            return False

        # --- PASO 3: EJECUTAR (ACTION) ---
        print(f"\n🔹 [PASO 3/3] Despachando tareas al Executor...")
        if not action_03(sat_id, product_id, year, day, fnp_tag):
            print(f"❌ {ctx} Falló el despacho de ejecución.")
            return False

        # --- CIERRE ---
        total_time = round(time.time() - start_all, 2)
        print("\n" + f" ✨ PIPELINE FINALIZADO CON ÉXITO EN {total_time}s ".center(80, "=") + "\n")
        return True

    except Exception as e:
        print(f"\n💥 {ctx} ERROR CRÍTICO NO CONTROLADO: {e}")
        return False

# =============================================================================
# MAIN: EJECUCIÓN DEL DÍA COMPLETO
# =============================================================================
if __name__ == "__main__":
    # Parámetros para procesar todo el día 003 de LSTF
    params = {
        "sat_id": "19", 
        "product_id": "ABI-L2-LSTF", 
        "year": "2026", 
        "day": "003", 
        "fnp_tag": "fnp01"
    }
    
    run_subtask_01(**params)

In [4]:
# =============================================================================
# FILE PATH: legion_goes/tasks/task03_processing/subtask01_proc_single/actions/action03_run_plan_proc_single.py
# Version: 3.6.0 (Feature: Immediate Audit on Success)
# =============================================================================
import os
from pathlib import Path

# --- ABSOLUTE IMPORTS ---
from legion_goes.code.python_sp.f99_common.load_dict_plan_from_json_file import load_dict_plan_from_json_file
from legion_goes.code.python_sp.f02_processing.sp001_single.utils.generate_plan_proc_single_json_file_path import generate_plan_proc_single_json_file_path

# Importamos el Executor
from legion_goes.code.python_sp.f02_processing.sp001_single.f02_auto_processing.fn02_run_executor.run_executor import run_executor

# Importamos la Action 02 para actualizar el diccionario tras cada éxito
from legion_goes.tasks.task03_processing.subtask01_proc_single.actions.action02_update_json_plan_proc_single import run_action as run_audit

def run_action(sat_id: str, product_id: str, year: str, day: str, fnp_tag: str = "fnp01") -> bool:
    """
    Action 03: Despachador. 
    Tras cada ejecución exitosa, llama a la Action 02 para sincronizar el JSON.
    """
    ctx = "[Action03 - Dispatcher]"
    
    # 1. Localizar el plan JSON
    path_plan = generate_plan_proc_single_json_file_path(
        sat_id=sat_id, product_id=product_id, year=year, day=day, fnp_tag=fnp_tag
    )

    # 2. Cargar el contenido
    plan_data = load_dict_plan_from_json_file(path_json=str(path_plan))
    if not plan_data:
        print(f"❌ {ctx} Error: No se pudo cargar el plan.")
        return False

    inventory = plan_data.get("inventory", {})
    
    # 3. Filtrar pendientes
    pending_fids = [
        fid for fid, item in inventory.items() 
        if item["tracking"].get("is_ready_to_proc") and not item["tracking"].get("is_done_proc")
    ]

    if not pending_fids:
        print(f"☕ {ctx} Nada pendiente por procesar.")
        return True

    print(f"🚀 {ctx} Iniciando despacho de {len(pending_fids)} tareas...")

    # 4. BUCLE DE EJECUCIÓN + ACTUALIZACIÓN
    for i, fid in enumerate(pending_fids, 1):
        item = inventory[fid]
        nc_path = item["definition"]["input_info"]["soft"]["file_path"]

        print(f"\n📦 [{i}/{len(pending_fids)}] Ejecutando: {fid}")
        
        try:
            # EJECUCIÓN
            success = run_executor(nc_path=nc_path, fnp_tag=fnp_tag)
            
            if success:
                print(f"   ✅ Executor exitoso. Sincronizando Plan JSON...")
                # --- EL UPDATE CLAVE ---
                # Llamamos a la Action 02 para que verifique el disco y marque como DONE
                run_audit(sat_id=sat_id, product_id=product_id, year=year, day=day, fnp_tag=fnp_tag)
            else:
                print(f"   ⚠️  El Executor reportó fallos en {fid}. No se actualizó el estado.")
                
        except Exception as e:
            print(f"   ❌ [CRITICAL] Error en {fid}: {e}")
            continue

    print(f"\n✨ {ctx} Proceso terminado y JSON sincronizado.")
    return True

if __name__ == "__main__":
    params = {
        "sat_id": "19", 
        "product_id": "ABI-L2-LSTF", 
        "year": "2026", 
        "day": "003", 
        "fnp_tag": "fnp01"
    }
    run_action(**params)

🚀 [Action03 - Dispatcher] Iniciando despacho de 24 tareas...

📦 [1/24] Ejecutando: proc_single_01

------------------------------------------------------------
📦 [COLLECTOR] Building plan: ABI-L2-LSTF | s202600300
------------------------------------------------------------
  ✅ Step 1/3: Pack01 (Science) collected.
  ✅ Step 2/3: Pack02 (Gallery) collected.
  ✅ Step 3/3: Pack03 (Metadata) collected.

🚀 Collector finished. Total steps in plan: 3
------------------------------------------------------------


 🚀 PROCESANDO: OR_ABI-L2-LSTF-M6_G19_s20260030000228_e20260030009536_c20260030015142.nc 

▶️  [1/3] Standard Processing: ABI-L2-LSTF

      [Step 01/06] 🛰️  Loading LST Scene... Done.
      [Step 02/06] 📸  Saving Native PNGs... Done.
      [Step 03/06] 🔄  Resampling to WGS84 Area... Done.
      [Step 04/06] 💾  Saving WGS84 GeoTIFFs... Done.
      [Step 05/06] 📸  Saving WGS84 PNGs... Done.
      [Step 06/06] 📝  Cleaning... Done.

▶️  [2/3] Gallery Strip Generation
      📸 [GALLERY] Cre